# RAG from Scratch to Production — Episode 3/20
## Parsing PDF, DOCX and TXT: the bugs nobody sees

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakkulaMahendhar/rag-from-scratch/blob/main/notebooks/03_parsing.ipynb)

So far our documents were neat Python strings. Real ones arrive as **PDF, Word and text files**, and step 0 of every RAG system is turning those files into plain text.

It sounds boring. It is also where the **quietest bugs** live: when parsing loses a fact, nothing crashes. The bot just says *"I don't have enough information"*, and you go looking for the problem in retrieval or the LLM.

**In this episode:**
1. Create 4 realistic files: a PDF, a scanned PDF, a Word file with a table, and a text file saved on Windows.
2. Parse them with the **exact parser from the [production app](https://github.com/JakkulaMahendhar/rag-chatbot)** (`app/services/parser.py`).
3. Watch it fail 4 different ways, and see how each failure reaches the user.
4. Fix all four, and get the right answers back.

## 0. Setup

- **PyMuPDF** (`fitz`) reads PDFs. **python-docx** reads Word files.
- The LLM is only used at the end, to show what the user would see. Same `generate()` as earlier episodes: Gemini (free key, add it as a Colab secret named `GEMINI_API_KEY`) or local Ollama.

In [ ]:
%pip install -q pymupdf python-docx google-generativeai requests

In [1]:
import os, re, requests
from pathlib import Path

import fitz                  # PyMuPDF
from docx import Document    # python-docx

PROVIDER = os.getenv("LLM_PROVIDER", "gemini")   # "gemini" or "ollama"
GEMINI_MODEL = "gemini-3.6-flash"
OLLAMA_MODEL = "llama3.1"


def _gemini_key():
    try:
        from google.colab import userdata  # Colab secrets
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        return os.getenv("GEMINI_API_KEY") or __import__("getpass").getpass("GEMINI_API_KEY: ")


if PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=_gemini_key())
    _model = genai.GenerativeModel(GEMINI_MODEL)


def generate(prompt: str) -> str:
    """One interface, two providers. Temperature 0 so results are repeatable."""
    if PROVIDER == "gemini":
        return _model.generate_content(
            prompt, generation_config={"temperature": 0}
        ).text.strip()
    r = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False,
              "options": {"temperature": 0}},
        timeout=300,
    )
    r.raise_for_status()
    return r.json()["response"].strip()


print(f"Using {PROVIDER}")

Using ollama


## 1. Create 4 realistic files

Everything is generated here, so the notebook runs anywhere. Each file hides one classic problem:

| File | What's inside | The trap |
|---|---|---|
| `handbook.pdf` | 2 pages of policy text | A header and page number on every page, and a sentence that continues onto page 2 |
| `parental_leave.pdf` | A **scanned** page | It is a picture of text. There is no text inside the file at all |
| `benefits.docx` | Paragraphs **and a table** | The amounts are only in the table |
| `cafeteria.txt` | A short note, saved on Windows | Not UTF-8. The `€` sign is stored as a Windows-1252 byte |

In [2]:
DOCS = Path("sample_docs")
DOCS.mkdir(exist_ok=True)

# 1. A normal 2-page PDF, with a header + footer on every page
PAGES = [
    "Remote Work\n"
    "Employees may work remotely up to 2 days per week. Wednesday is a mandatory "
    "in-office day for all teams. Remote days must be logged in the HR portal by",
    "Friday of the previous week.\n\n"
    "Annual Leave\n"
    "Full-time employees receive 24 days of paid annual leave per calendar year. "
    "A maximum of 5 unused days may be carried over into the next year.",
]
pdf = fitz.open()
for i, body in enumerate(PAGES, start=1):
    page = pdf.new_page()
    page.insert_text((72, 40), "NovaTech Solutions - CONFIDENTIAL", fontsize=9)
    page.insert_textbox(fitz.Rect(72, 80, 520, 760), body, fontsize=12)
    page.insert_text((270, 810), f"Page {i} of {len(PAGES)}", fontsize=9)
pdf.save(DOCS / "handbook.pdf")

# 2. A "scanned" PDF: render text to an image, then save only the image
original = fitz.open()
page = original.new_page()
page.insert_textbox(fitz.Rect(72, 72, 520, 400),
                    "Parental Leave\nNew parents receive 16 weeks of fully paid leave.",
                    fontsize=16)
picture = page.get_pixmap(dpi=100)
scanned = fitz.open()
scanned.new_page().insert_image(fitz.Rect(0, 0, 595, 842), stream=picture.tobytes("png"))
scanned.save(DOCS / "parental_leave.pdf", deflate=True)

# 3. A Word file where the important numbers live in a table
doc = Document()
doc.add_heading("Employee Benefits", level=1)
doc.add_paragraph("NovaTech pays the following monthly benefits to every employee.")
table = doc.add_table(rows=1, cols=2)
table.rows[0].cells[0].text, table.rows[0].cells[1].text = "Benefit", "Amount"
for benefit, amount in [("Gym membership", "50 USD per month"),
                        ("Phone bill", "30 USD per month"),
                        ("Internet at home", "25 USD per month")]:
    row = table.add_row().cells
    row[0].text, row[1].text = benefit, amount
doc.add_paragraph("Benefits are paid together with the next salary.")
doc.save(DOCS / "benefits.docx")

# 4. A text file saved by a Windows tool (Windows-1252, not UTF-8)
(DOCS / "cafeteria.txt").write_bytes(
    "Cafeteria\nLunch vouchers are worth 8 € per working day.\n".encode("cp1252")
)

for f in sorted(DOCS.iterdir()):
    print(f"{f.name:<20} {f.stat().st_size:>7,} bytes")

benefits.docx         36,892 bytes
cafeteria.txt             56 bytes
handbook.pdf           2,240 bytes
parental_leave.pdf    14,238 bytes


## 2. The production parser, as it is today

This is `app/services/parser.py` from the [production app](https://github.com/JakkulaMahendhar/rag-chatbot), unchanged except for the name. One entry point, one small function per format:

In [3]:
def parse_v1(path: Path) -> str:
    ext = path.suffix.lower()
    if ext == ".pdf":
        document = fitz.open(path)
        text = ""
        for page in document:
            text += page.get_text()
        document.close()
        return text
    if ext == ".docx":
        document = Document(path)
        return "\n".join(p.text for p in document.paragraphs)
    if ext == ".txt":
        with open(path, "r", encoding="utf-8") as file:
            return file.read()
    raise ValueError("Unsupported document type")

Let's run it on all 4 files, the way the background worker would. A parse error marks the document `failed`. Otherwise it is `completed`.

In [4]:
def ingest(parser) -> dict[str, str]:
    """Parse every file like the worker does, and print a status line per file."""
    texts = {}
    for path in sorted(DOCS.iterdir()):
        try:
            text = parser(path)
            texts[path.name] = text
            print(f"✅ {path.name:<20} completed   {len(text):>4} characters")
        except Exception as e:
            print(f"❌ {path.name:<20} failed      {type(e).__name__}: {str(e)[:60]}")
    return texts


v1 = ingest(parse_v1)

✅ benefits.docx        completed    130 characters
❌ cafeteria.txt        failed      UnicodeDecodeError: 'utf-8' codec can't decode byte 0x80 in position 37: invalid
✅ handbook.pdf         completed    441 characters
✅ parental_leave.pdf   completed      0 characters


Three green ticks and one crash. That looks like **1 problem**. There are **4**. Let's look at what the parser actually produced.

### Failure 1: the PDF is full of noise

In [5]:
print(v1["handbook.pdf"])

NovaTech Solutions - CONFIDENTIAL
Remote Work
Employees may work remotely up to 2 days per week. Wednesday is a mandatory
in-office day for all teams. Remote days must be logged in the HR portal by
Page 1 of 2
NovaTech Solutions - CONFIDENTIAL
Friday of the previous week.
Annual Leave
Full-time employees receive 24 days of paid annual leave per calendar year. A
maximum of 5 unused days may be carried over into the next year.
Page 2 of 2



- The header **"NovaTech Solutions - CONFIDENTIAL"** and **"Page 1 of 2"** are now part of the text. On a 300-page PDF, that's 600 lines of noise copied into your chunks.
- Worse, they land **in the middle of a sentence**: *"must be logged in the HR portal by"* … *"Page 1 of 2"* … *"NovaTech Solutions - CONFIDENTIAL"* … *"Friday of the previous week."* Split that into chunks and the deadline can end up separated from the sentence it belongs to.
- Lines also break wherever the PDF wrapped them, not where sentences end.

### Failure 2: the scanned PDF is "completed", with nothing in it

In [6]:
print(repr(v1["parental_leave.pdf"]))

''


**0 characters, and status `completed`.** A scanned page is a photo of text, so there is no text to extract. No error, no warning. The parental leave policy is uploaded, marked done, and **can never be found**.

### Failure 3: the Word table disappeared

In [7]:
print(v1["benefits.docx"])

Employee Benefits
NovaTech pays the following monthly benefits to every employee.
Benefits are paid together with the next salary.


`document.paragraphs` only returns paragraphs. **Tables are stored separately in a .docx file**, so every amount (gym 50 USD, phone 30 USD, internet 25 USD) is gone. In real company documents, tables are where the numbers live: prices, limits, dates, contacts.

### Failure 4: the text file crashed

`open(path, encoding="utf-8")` assumes UTF-8. Many files saved by Windows tools and older systems use **Windows-1252**, where `€` is a single byte (`0x80`) that isn't valid UTF-8. This one at least fails loudly. The whole file is rejected because of one symbol.

## 3. What the user sees

The user doesn't see any of this. They just ask questions. Here is the Episode 1 RAG pipeline, with simple sentence-based chunks (Episode 4 covers chunking properly):

In [8]:
STOPWORDS = {"a", "an", "the", "is", "are", "and", "or", "of", "to", "in", "on",
             "at", "for", "per", "what", "how", "many", "much", "does", "do",
             "i", "my", "can", "get", "it", "be", "must", "by", "with", "who"}

def words(text: str) -> set[str]:
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOPWORDS}


def make_chunks(texts: dict[str, str], size: int = 3) -> list[str]:
    """Split each document into sentences, then group every `size` sentences."""
    chunks = []
    for text in texts.values():
        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n{2,}", text) if s.strip()]
        chunks += [" ".join(sentences[i:i + size]) for i in range(0, len(sentences), size)]
    return chunks


def retrieve(question: str, chunks: list[str]) -> str:
    q = words(question)
    score, best = max(((len(q & words(c)), c) for c in chunks), default=(0, ""))
    return best if score > 0 else ""


FALLBACK = "I don't have enough information from the provided documents."

def rag(question: str, texts: dict[str, str]) -> str:
    context = retrieve(question, make_chunks(texts))
    prompt = f"""Answer only using the context below. If the answer is not in the
context, respond exactly: "{FALLBACK}"

Context:
{context or "(no relevant context found)"}

Question: {question}
Answer:"""
    return generate(prompt)


QUESTIONS = [
    "How much is the gym membership benefit?",
    "How many weeks of parental leave do new parents get?",
    "How much are lunch vouchers worth?",
    "When must remote days be logged in the HR portal?",
]

for q in QUESTIONS:
    print(f"Q: {q}\nA: {rag(q, v1)}\n")

Q: How much is the gym membership benefit?
A: I don't have enough information from the provided documents.



Q: How many weeks of parental leave do new parents get?
A: I don't have enough information from the provided documents.



Q: How much are lunch vouchers worth?
A: I don't have enough information from the provided documents.



Q: When must remote days be logged in the HR portal?
A: Friday of the previous week.



Every one of these answers is in the files we uploaded. The bot can't answer the first three because **they never made it through parsing**. To the user, the bot just looks weak.

The last one usually still works: the LLM reads through the header and footer noise. Usually.

## 4. Fix it: `parse_v2`

One fix per failure:

| Failure | Fix |
|---|---|
| PDF noise | Read page by page. Drop lines that repeat on every page (headers) and `Page N of M` (footers). Join lines that were only wrapped |
| Scanned PDF | If a page has an image but no text, **fail loudly**: "needs OCR". Never mark an empty document `completed` |
| Word tables | Walk the document **in order**, paragraphs and tables, and turn each table row into a sentence: `Gym membership: 50 USD per month.` |
| Encoding | Try UTF-8 first (with or without a BOM), then fall back to Windows-1252 |

In [9]:
def parse_pdf(path: Path) -> str:
    with fitz.open(path) as document:
        pages = [page.get_text() for page in document]
        has_images = [bool(page.get_images()) for page in document]

    # Scanned pages: an image with no text layer. Fail loudly instead of returning "".
    for number, (text, image) in enumerate(zip(pages, has_images), start=1):
        if not text.strip() and image:
            raise ValueError(f"page {number} is a scanned image with no text: needs OCR")

    # Headers: lines that appear on every page. Footers: "Page N of M".
    lines_per_page = [{line.strip() for line in p.splitlines() if line.strip()} for p in pages]
    repeated = set.intersection(*lines_per_page) if len(pages) > 1 else set()
    page_number = re.compile(r"^page \d+( of \d+)?$", re.IGNORECASE)

    kept = [
        line for p in pages for line in p.splitlines()
        if line.strip() not in repeated and not page_number.match(line.strip())
    ]
    # Rejoin wrapped lines: a long line that doesn't end a sentence continues on
    # the next line (even across a page break). Short lines are headings: keep them.
    merged = []
    for line in (l.strip() for l in kept):
        if not line:
            merged.append("")
        elif merged and len(merged[-1]) > 40 and not merged[-1].endswith((".", "!", "?", ":")):
            merged[-1] += " " + line
        else:
            merged.append(line)
    return "\n".join(merged).strip()


def parse_docx(path: Path) -> str:
    document = Document(path)
    blocks = []
    # iter_inner_content() yields paragraphs AND tables, in document order.
    for block in document.iter_inner_content():
        if hasattr(block, "rows"):                      # a table
            header = [cell.text.strip() for cell in block.rows[0].cells]
            for row in block.rows[1:]:
                cells = [cell.text.strip() for cell in row.cells]
                blocks.append(f"{cells[0]}: " + ", ".join(cells[1:]) + ".")
            blocks.append("")
        elif block.text.strip():                        # a paragraph
            blocks.append(block.text.strip())
    return "\n".join(blocks).strip()


def parse_txt(path: Path) -> str:
    raw = path.read_bytes()
    for encoding in ("utf-8-sig", "cp1252"):   # utf-8-sig also strips a BOM if present
        try:
            return raw.decode(encoding)
        except UnicodeDecodeError:
            continue
    return raw.decode("utf-8", errors="replace")


def parse_v2(path: Path) -> str:
    ext = path.suffix.lower()
    if ext == ".pdf":
        return parse_pdf(path)
    if ext == ".docx":
        return parse_docx(path)
    if ext == ".txt":
        return parse_txt(path)
    raise ValueError("Unsupported document type")


v2 = ingest(parse_v2)

✅ benefits.docx        completed    231 characters
✅ cafeteria.txt        completed     56 characters
✅ handbook.pdf         completed    348 characters
❌ parental_leave.pdf   failed      ValueError: page 1 is a scanned image with no text: needs OCR


Now the status lines tell the truth: **3 files completed with real content, 1 failed with a reason a human can act on.**

Let's check each fix:

In [10]:
for name in ["handbook.pdf", "benefits.docx", "cafeteria.txt"]:
    print(f"── {name} " + "─" * (40 - len(name)))
    print(v2[name], "\n")

── handbook.pdf ────────────────────────────
Remote Work
Employees may work remotely up to 2 days per week. Wednesday is a mandatory in-office day for all teams. Remote days must be logged in the HR portal by Friday of the previous week.
Annual Leave
Full-time employees receive 24 days of paid annual leave per calendar year. A maximum of 5 unused days may be carried over into the next year. 

── benefits.docx ───────────────────────────
Employee Benefits
NovaTech pays the following monthly benefits to every employee.
Gym membership: 50 USD per month.
Phone bill: 30 USD per month.
Internet at home: 25 USD per month.

Benefits are paid together with the next salary. 

── cafeteria.txt ───────────────────────────
Cafeteria
Lunch vouchers are worth 8 € per working day.
 



- **PDF:** no header, no page numbers, and *"logged in the HR portal by Friday of the previous week"* is one sentence again.
- **Word:** each table row is now a sentence that still makes sense on its own, even after chunking: `Gym membership: 50 USD per month.`
- **Text:** `8 €` decoded correctly.

## 5. Ask again

In [11]:
for q in QUESTIONS:
    print(f"Q: {q}\nA: {rag(q, v2)}\n")

Q: How much is the gym membership benefit?
A: 50 USD per month.



Q: How many weeks of parental leave do new parents get?
A: I don't have enough information from the provided documents.



Q: How much are lunch vouchers worth?
A: Lunch vouchers are worth 8 € per working day.

Q: When must remote days be logged in the HR portal?
A: By Friday of the previous week.



- **Gym, lunch vouchers, HR portal:** answered from the fixed text.
- **Parental leave:** still *"I don't have enough information"*, but now **you know why**. The upload says `failed: page 1 is a scanned image with no text: needs OCR`, so someone can run OCR (for example Tesseract, or a vision-capable LLM) or upload a text version. Before, it was marked `completed` and nobody would ever find out.

## Recap

| File | Production parser today | `parse_v2` |
|---|---|---|
| Normal PDF | ⚠️ Header/footer noise inside sentences | ✅ Clean text, sentences rejoined |
| Scanned PDF | ❌ `completed` with **0 characters**, silently | ✅ `failed: needs OCR`, loudly |
| Word with a table | ❌ Table silently dropped | ✅ Each row becomes a sentence |
| Windows text file | ❌ Crash on `€` | ✅ Falls back to Windows-1252 |

**The rule:** after parsing, *look at the text*. If a fact isn't in the parsed text, no retriever, reranker or LLM can ever find it.

**Next, Episode 4:** we have clean text. Now, how do we cut it into chunks without cutting answers in half?

⭐ Notebooks: https://github.com/JakkulaMahendhar/rag-from-scratch
🏗️ Full production app: https://github.com/JakkulaMahendhar/rag-chatbot
📸 Instagram: [@codingwithmahi](https://instagram.com/codingwithmahi)